<a href="https://colab.research.google.com/github/heavenmaker024/114-2PL-Repo61271012H/blob/main/HW4_PTT_GoogleSheet_RAG_%E6%B5%A901%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 PTT movie 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [ ]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


In [ ]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。


In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Hg-6uiw7Qr0HtqSzP8yBzv7dBH9uEE9gMgPNSAXRtR8/edit?gid=0#gid=0"
PTT_WORKSHEET_NAME = "ptt_movie_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/car/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"


## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [ ]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：HW4_文字資料小分析
🔗 https://docs.google.com/spreadsheets/d/1Hg-6uiw7Qr0HtqSzP8yBzv7dBH9uEE9gMgPNSAXRtR8/edit?gid=0#gid=0


In [ ]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")

✅ 已準備 worksheet：ptt_movie_posts


## 3. PTT movie 爬蟲

這段只負責爬 PTT，不碰 RAG。資料會先存在 `new_posts_df`。


In [ ]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_MOVIE_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 PTT 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [ ]:
# 你可以調整 pages，例如 pages=1 先測試，確認成功後再改成 3 或 5
new_posts_df = crawl_ptt_movie(pages=2, delay=1.0)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/2: https://www.ptt.cc/bbs/car/index.html


ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [ ]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 清掉沒有內容的文章
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()


In [ ]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [ ]:
api_key = "Geminiapikey"
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")

In [ ]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

## 7. 快速測試


In [ ]:
# 發生 NameError 通常是因為定義函式的單元格尚未執行
# 請確保已執行上方定義 query_rag 與 retrieve_docs 的單元格

try:
    question = input("請輸入問題：")
    answer = query_rag(question, k=3)
    print(answer)
except NameError as e:
    print(f"❌ 錯誤：{e}")
    print("請先執行上方的單元格以定義 'query_rag' 函式。")

請輸入問題：電影
❌ 錯誤：name 'rag_index' is not defined
請先執行上方的單元格以定義 'query_rag' 函式。


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。


In [1]:
# 1. 安裝必要套件
!pip -q install -U gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai pandas

import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe

# ==========================================
# 1. 系統設定與驗證
# ==========================================
print("🔄 正在驗證系統與金鑰...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

api_key = userdata.get("Geminiapikey")
if not api_key:
    raise ValueError("❌ 找不到 Colab Secret 金鑰。請先在左側密碼本新增 'Geminiapikey'。")
genai.configure(api_key=api_key)

SHEET_URL = "https://docs.google.com/spreadsheets/d/1Hg-6uiw7Qr0HtqSzP8yBzv7dBH9uEE9gMgPNSAXRtR8/edit?gid=0#gid=0"
PTT_WORKSHEET_NAME = "ptt_car_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = ["post_id", "title", "url", "date", "author", "nrec", "created_at", "fetched_at", "content"]
PTT_CAR_INDEX = "https://www.ptt.cc/bbs/car/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")

# ==========================================
# 2. 試算表操作模組
# ==========================================
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update(values=[header], range_name="A1")
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update(values=[header], range_name="A1")
    elif values[0] != header:
        ws.clear()
        ws.update(values=[header], range_name="A1")
    return ws

def read_sheet_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns: df[col] = ""
    return df[header].fillna("")

def write_sheet_df(ws, df, header):
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns: df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("")
    for c in df_out.columns: df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)

ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)

# ==========================================
# 3. PTT 爬蟲模組 (強化清洗)
# ==========================================
def now_iso():
    return datetime.now().isoformat(timespec="seconds")

def get_soup(url):
    resp = requests.get(url, timeout=20, headers={"User-Agent": USER_AGENT}, cookies=PTT_COOKIES)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None

def parse_nrec(nrec_span):
    if not nrec_span: return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆": return 100
    if txt.startswith("X"):
        try: return -int(txt[1:])
        except: return -10
    try: return int(txt)
    except: return 0

def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a: continue
        title = a.get_text(strip=True)
        # 過濾掉公告
        if "公告" in title: continue

        posts.append({
            "title": title,
            "url": urljoin("https://www.ptt.cc", a.get("href")),
            "author": item.select_one("div.author").get_text(strip=True) if item.select_one("div.author") else "",
            "date": item.select_one("div.date").get_text(strip=True) if item.select_one("div.date") else "",
            "nrec": parse_nrec(item.select_one("div.nrec span")),
        })
    return posts

def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main: return "", ""

    created_at = ""
    for m in main.select("div.article-metaline"):
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta, 推文, 發信站等雜訊
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push, span.f2"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at

def make_post_id(url):
    return url.rstrip("/").split("/")[-1].replace(".html", "")

def crawl_and_update_ptt_car(pages=2, delay=0.5):
    """爬取最新文章並與試算表資料合併去重"""
    print(f"\n🕷️ 開始爬取 PTT 汽車版最新 {pages} 頁文章...")
    all_rows = []
    index_url = PTT_CAR_INDEX

    for page in range(int(pages)):
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)

                # 若內容太短則跳過
                if len(content) < 10: continue

                all_rows.append({
                    "post_id": make_post_id(p["url"]), "title": p["title"], "url": p["url"],
                    "date": p["date"], "author": p["author"], "nrec": p["nrec"],
                    "created_at": created_at, "fetched_at": now_iso(), "content": content,
                })
                time.sleep(delay)
            except Exception as e:
                pass

        prev_url = get_prev_index_url(index_soup)
        if not prev_url: break
        index_url = prev_url
        time.sleep(delay)

    new_df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次成功爬取 {len(new_df)} 篇有效文章。")

    # 讀取舊資料並合併去重
    print("🔄 正在與 Google Sheet 舊資料合併去重...")
    existing_df = read_sheet_df(ws_ptt, PTT_HEADER)
    combined_df = pd.concat([existing_df, new_df]).drop_duplicates(subset=["post_id"], keep="last")

    # 寫回 Google Sheet
    write_sheet_df(ws_ptt, combined_df, PTT_HEADER)
    print(f"✅ 寫入完成！目前資料庫共有 {len(combined_df)} 篇文章。")
    return combined_df

# 執行爬蟲更新 (預設爬 2 頁，你可以自己改)
rag_source_df = crawl_and_update_ptt_car(pages=2, delay=2)

# ==========================================
# 4. 建立 FAISS 向量索引
# ==========================================
# 清除空內容
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"\n📚 準備進入 RAG 系統的文章數：{len(rag_source_df)}")

print("⏳ 正在載入 Embedding 模型 (初次執行需下載)...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        text = f"標題：{title}\n作者：{str(row.get('author', ''))}\n日期：{str(row.get('date', ''))}\n內容：{str(row.get('content', ''))}"
        docs.append({"post_id": str(row.get("post_id", "")), "title": title, "url": str(row.get("url", "")), "text": text})
    return docs

def build_faiss_index(docs):
    if not docs: raise ValueError("沒有可建立索引的文件")
    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings

rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)
print(f"✅ RAG 索引建立完成！向量維度：{rag_embeddings.shape[1]}")

# ==========================================
# 5. Gemini 問答系統 (RAG 檢索生成)
# ==========================================
GEMINI_MODEL_NAME = "gemini-2.5-flash"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)

def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents: return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1: continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results

def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs: return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join([f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs])

    # 🚗 更新為 PTT 汽車版的專屬 Prompt
    prompt = f"""
你是一個根據 PTT 汽車版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

# ==========================================
# 6. 執行互動問答測試
# ==========================================
print("\n" + "="*50)
question = input("👉 請輸入問題 (例如：2026新款豐田有甚麼？)：")
print("\n🤖 AI 思考中...\n")
anwer = query_rag(question, k=3)
print("=" * 50)
print(answer)
print("=" * 50)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 27.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


🔄 正在驗證系統與金鑰...
✅ 已開啟試算表：HW4_文字資料小分析

🕷️ 開始爬取 PTT 汽車版最新 2 頁文章...


ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

In [4]:
# 1. 安裝必要套件 (精準鎖定版本以解決衝突，並使用最新的 google-genai)
!pip -q install pandas==2.2.2 requests==2.32.4 gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 google-genai gradio

import re
import time
import uuid
import random
from datetime import datetime
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

# 載入最新的 Google GenAI SDK
from google import genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe

# ==========================================
# 1. 系統設定與驗證
# ==========================================
print("🔄 正在驗證系統與金鑰...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

api_key = userdata.get("Geminiapikey")
if not api_key:
    raise ValueError("❌ 找不到 Colab Secret 金鑰。請先在左側密碼本新增 'Geminiapikey'。")

# 使用新版 SDK 初始化 Client
client = genai.Client(api_key=api_key)

SHEET_URL = "https://docs.google.com/spreadsheets/d/1Hg-6uiw7Qr0HtqSzP8yBzv7dBH9uEE9gMgPNSAXRtR8/edit?gid=0#gid=0"
PTT_WORKSHEET_NAME = "ptt_car_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = ["post_id", "title", "url", "date", "author", "nrec", "created_at", "fetched_at", "content"]
PTT_CAR_INDEX = "https://www.ptt.cc/bbs/car/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")

# ==========================================
# 2. 試算表操作模組
# ==========================================
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update(values=[header], range_name="A1")
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update(values=[header], range_name="A1")
    elif values[0] != header:
        ws.clear()
        ws.update(values=[header], range_name="A1")
    return ws

def read_sheet_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns: df[col] = ""
    return df[header].fillna("")

def write_sheet_df(ws, df, header):
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns: df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("")
    for c in df_out.columns: df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)

ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)

# ==========================================
# 3. 強化防禦 PTT 爬蟲模組 (解決被踢下線的問題)
# ==========================================
def now_iso():
    return datetime.now().isoformat(timespec="seconds")

# 建立具有重試機制的連線 Session
session = requests.Session()
# 當遇到 429(請求太多)、500、502、503、504 或連線中斷時，自動重試最多 5 次
retry = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)
session.headers.update({"User-Agent": USER_AGENT})
session.cookies.update(PTT_COOKIES)

def get_soup(url):
    # 【關鍵】加入 0.5 ~ 2 秒的隨機延遲，偽裝成真人在點擊網頁
    time.sleep(random.uniform(0.5, 2.0))
    resp = session.get(url, timeout=20)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None

def parse_nrec(nrec_span):
    if not nrec_span: return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆": return 100
    if txt.startswith("X"):
        try: return -int(txt[1:])
        except: return -10
    try: return int(txt)
    except: return 0

def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a: continue
        title = a.get_text(strip=True)
        if "公告" in title: continue

        posts.append({
            "title": title,
            "url": urljoin("https://www.ptt.cc", a.get("href")),
            "author": item.select_one("div.author").get_text(strip=True) if item.select_one("div.author") else "",
            "date": item.select_one("div.date").get_text(strip=True) if item.select_one("div.date") else "",
            "nrec": parse_nrec(item.select_one("div.nrec span")),
        })
    return posts

def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main: return "", ""

    created_at = ""
    for m in main.select("div.article-metaline"):
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    for node in main.select("div.article-metaline, div.article-metaline-right, div.push, span.f2"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at

def make_post_id(url):
    return url.rstrip("/").split("/")[-1].replace(".html", "")

def crawl_and_update_ptt_car(pages=2):
    print(f"\n🕷️ 開始爬取 PTT 汽車版最新 {pages} 頁文章...")
    all_rows = []
    index_url = PTT_CAR_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}...")
        try:
            index_soup = get_soup(index_url)
            post_list = extract_post_list(index_soup)

            for p in post_list:
                try:
                    article_soup = get_soup(p["url"])
                    content, created_at = clean_ptt_content(article_soup)

                    if len(content) < 10: continue

                    all_rows.append({
                        "post_id": make_post_id(p["url"]), "title": p["title"], "url": p["url"],
                        "date": p["date"], "author": p["author"], "nrec": p["nrec"],
                        "created_at": created_at, "fetched_at": now_iso(), "content": content,
                    })
                except Exception as e:
                    print(f"⚠️ 跳過文章：{p.get('title', '')} (原因：{e})")

            prev_url = get_prev_index_url(index_soup)
            if not prev_url: break
            index_url = prev_url

        except Exception as e:
            print(f"❌ 讀取列表頁失敗: {e}")
            break

    new_df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次成功爬取 {len(new_df)} 篇有效文章。")

    print("🔄 正在與 Google Sheet 舊資料合併去重...")
    existing_df = read_sheet_df(ws_ptt, PTT_HEADER)
    combined_df = pd.concat([existing_df, new_df]).drop_duplicates(subset=["post_id"], keep="last")

    write_sheet_df(ws_ptt, combined_df, PTT_HEADER)
    print(f"✅ 寫入完成！目前資料庫共有 {len(combined_df)} 篇文章。")
    return combined_df

# 執行爬蟲更新 (預設爬 2 頁)
rag_source_df = crawl_and_update_ptt_car(pages=2)

# ==========================================
# 4. 建立 FAISS 向量索引
# ==========================================
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"\n📚 準備進入 RAG 系統的文章數：{len(rag_source_df)}")

print("⏳ 正在載入 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        text = f"標題：{title}\n作者：{str(row.get('author', ''))}\n日期：{str(row.get('date', ''))}\n內容：{str(row.get('content', ''))}"
        docs.append({"post_id": str(row.get("post_id", "")), "title": title, "url": str(row.get("url", "")), "text": text})
    return docs

def build_faiss_index(docs):
    if not docs: raise ValueError("沒有可建立索引的文件")
    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings

rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)
print(f"✅ RAG 索引建立完成！")

# ==========================================
# 5. Gemini 問答系統 (使用新版 google-genai)
# ==========================================
GEMINI_MODEL_NAME = "gemini-2.5-flash"

def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents: return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1: continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results

def query_rag(question, k=3, max_retries=3):
    docs = retrieve_docs(question, k=k)
    if not docs: return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join([f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs])

    prompt = f"""
你是一個根據 PTT 汽車版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    # 加入自動重試機制
    for attempt in range(max_retries):
        try:
            # 使用新版 SDK 呼叫 API
            response = client.models.generate_content(
                model=GEMINI_MODEL_NAME,
                contents=prompt
            )
            return response.text

        except Exception as e:
            error_msg = str(e)
            # 如果是 503 或 429 忙碌錯誤，則等待後重試
            if "503" in error_msg or "UNAVAILABLE" in error_msg or "429" in error_msg:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt  # 等待 1秒, 2秒...
                    print(f"⚠️ Google 伺服器忙碌中，等待 {wait_time} 秒後自動重試 (第 {attempt+1}/{max_retries} 次)...")
                    time.sleep(wait_time)
                else:
                    return f"❌ 抱歉，AI 伺服器目前大塞車，已自動重試 {max_retries} 次皆失敗，請稍後再試。"
            else:
                # 若是其他嚴重錯誤，直接回報
                return f"❌ 發生未知的 AI 錯誤：{error_msg}"
# ==========================================
# 6. 執行互動問答測試
# ==========================================
print("\n" + "="*50)
question = input("👉 請輸入問題 (例如：2026新款豐田有甚麼？)：")
print("\n🤖 AI 思考中...\n")
answer = query_rag(question, k=3)
print("=" * 50)
print(answer)
print("=" * 50)

🔄 正在驗證系統與金鑰...
✅ 已開啟試算表：HW4_文字資料小分析

🕷️ 開始爬取 PTT 汽車版最新 2 頁文章...
📄 正在讀取列表頁 1/2...
📄 正在讀取列表頁 2/2...
✅ 本次成功爬取 29 篇有效文章。
🔄 正在與 Google Sheet 舊資料合併去重...


/tmp/ipykernel_639/3492430026.py:87: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_out = df_out[header].infer_objects(copy=False).fillna("")


✅ 寫入完成！目前資料庫共有 29 篇文章。

📚 準備進入 RAG 系統的文章數：29
⏳ 正在載入 Embedding 模型...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RAG 索引建立完成！

👉 請輸入問題 (例如：2026新款豐田有甚麼？)：給我所有有關rav4的文章

🤖 AI 思考中...

⚠️ Google 伺服器忙碌中，等待 1 秒後自動重試 (第 1/3 次)...
⚠️ Google 伺服器忙碌中，等待 2 秒後自動重試 (第 2/3 次)...
❌ 抱歉，AI 伺服器目前大塞車，已自動重試 3 次皆失敗，請稍後再試。
